# L4a: Graph and Tree Representations

A graph represents objects and their connections. We call the objects *vertices* and the connections *edges*. Roads between cities, transformations between chemical species, and calls between functions can all be represented as graphs. These applications describe different systems but lead to common questions: which objects are connected, how can we move between them, and how should we store those connections for computation?

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Describe graph structure and measures:__ Define vertices, edges, walks, paths, cycles, and connectivity, including weak and strong connectivity in directed graphs. Calculate vertex degrees from the edges and graph density from vertex and edge counts.
> * __Characterize complete graphs, bipartite graphs, and trees:__ Explain each family's defining properties and use edge counts, coloring, and path properties to determine whether a graph belongs to it.
> * __Compare graph representations:__ Construct adjacency lists and adjacency matrices from an edge list. Explain their storage and access costs, and choose a representation based on graph density and the operations a computation requires.

In this lecture, we develop the language to describe graphs and examine three important graph families: complete graphs, bipartite graphs, and trees. We then compare edge lists, adjacency matrices, and adjacency lists. The choice of representation determines the memory required to store a graph and the work needed to find an edge or visit a vertex's neighbors.

Let's get started!

___


## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package and the `Test` standard library used to check our calculations.

Let's set up our code environment:


In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

See the [Julia documentation](https://docs.julialang.org/en/v1/) for language details and the [`Test` documentation](https://docs.julialang.org/en/v1/stdlib/Test/) for the checks used here. The graph functions are defined in [`GraphRepresentation.jl`](../../../code/src/GraphRepresentation.jl).

Our worked example uses [`data/SimpleGraph.txt`](data/SimpleGraph.txt), which contains seven weighted, directed edges connecting six vertices. We use this graph to construct and compare an adjacency list and an adjacency matrix. The dataset is described in [`data/README.md`](data/README.md).

___


## Simple Graphs

A graph describes which vertices are joined by edges. Before we compare ways to store these connections, let's establish the notation and explain how we can move through a graph.

> __Simple graph:__
>
> A simple graph $\mathcal{G}=(\mathcal{V},\mathcal{E})$ consists of a vertex set $\mathcal{V}$ and an edge set $\mathcal{E}$. The word *simple* rules out self-loops, which connect a vertex to itself, and repeated parallel edges.
>
> * In an __undirected graph__, an edge is an unordered pair $\{u,v\}$ of distinct vertices and can be followed in either direction.
> * In a __directed graph__, an edge is an ordered pair $(u,v)$ with $u\neq v$. It points from $u$ to $v$; the reverse edge $(v,u)$ is a different edge and may not exist.

For example, in a social network, person A may follow person B without B following A. Edges may also carry a __weight__, such as distance, capacity, or cost, whether the graph is directed or undirected.

__How do we describe movement through a graph?__ A __walk__ is a sequence of vertices $v_0,v_1,\ldots,v_k$ in which each consecutive pair is joined by an edge. In a directed graph, the walk must follow the edge directions. A __path__ is a walk with no repeated vertices. A __cycle__ is a closed walk containing at least one edge that repeats no edge and no vertex except its starting vertex, $v_0=v_k$.

These definitions let us describe __connectivity__:

* An undirected graph is __connected__ when a path joins every pair of vertices.
* A directed graph is __weakly connected__ when it becomes connected after edge directions are ignored.
* A directed graph is __strongly connected__ when every ordered pair $(u,v)$ has a directed path from $u$ to $v$.

<div>
    <center>
        <img src="figs/Fig-General-Graph-Schematic.svg" width="980" alt="Left: an undirected graph with six vertices and seven weighted edges, with the degree of vertex 4 marked. Right: the same vertices and edges drawn as a directed acyclic graph, with the in-degree of vertex 2 and the out-degree of vertex 4 marked."/>
    </center>
</div>

The undirected graph is connected; its directed counterpart is weakly but not strongly connected. We can reach vertex 5 from vertex 0, but cannot return because vertex 5 has no outgoing edge. With no directed cycles, it is a __directed acyclic graph__. Next, we use vertex degrees and other measures to quantify graph structure.


### Graph Measures

The __degree__ $\deg(v_i)$ of a vertex $v_i\in\mathcal{V}$ counts the edges that touch it. In a directed graph, we distinguish:

* __In-degree__ $\deg^{\mathrm{in}}(v_i)$: the number of edges pointing into $v_i$.
* __Out-degree__ $\deg^{\mathrm{out}}(v_i)$: the number of edges pointing out of $v_i$.

The total degree is $\deg(v_i)=\deg^{\mathrm{in}}(v_i)+\deg^{\mathrm{out}}(v_i)$.

__How are vertex degrees related to the number of edges?__ Let $n=|\mathcal{V}|$ and $m=|\mathcal{E}|$. Summing the degrees in an undirected graph counts both endpoints of every edge. In a directed graph, each edge contributes once to an in-degree and once to an out-degree.

> __Handshaking identities:__
>
> $$
> \begin{aligned}
> \sum_{v_i\in\mathcal{V}}\deg(v_i)&=2m
> &&\text{(undirected)},\\
> \sum_{v_i\in\mathcal{V}}\deg^{\mathrm{in}}(v_i)
> &=\sum_{v_i\in\mathcal{V}}\deg^{\mathrm{out}}(v_i)=m
> &&\text{(directed)}.
> \end{aligned}
> $$

For an undirected graph with $n\geq1$, dividing the degree sum by the number of vertices gives the __average degree__:
$$
\bar d(\mathcal{G})
=\frac{1}{n}\sum_{v_i\in\mathcal{V}}\deg(v_i)
=\frac{2m}{n}.
$$
The average lies between the __minimum degree__ $\delta(\mathcal{G})$ and __maximum degree__ $\Delta(\mathcal{G})$:
$$
\delta(\mathcal{G})\leq\bar d(\mathcal{G})\leq\Delta(\mathcal{G}).
$$
An undirected graph is __regular__ of degree $r$ when every vertex has degree $r$. In that case, the minimum, average, and maximum degrees all equal $r$.

__How many of the possible edges are present?__ For an undirected simple graph, each of the $n$ vertices can connect to $n-1$ others. Dividing by two avoids counting each edge twice. For a directed simple graph, the two directions represent different edges, so no division is needed.

> __Graph density:__
>
> For $n\geq2$, the maximum edge count and density are
> $$
> |\mathcal{E}|_{\max}=
> \begin{cases}
> n(n-1)/2 & \text{undirected},\\
> n(n-1) & \text{directed},
> \end{cases}
> \qquad
> \rho(\mathcal{G})=\frac{m}{|\mathcal{E}|_{\max}}.
> $$
>
> Density measures the fraction of possible edges that are present. A graph with $\rho$ close to 1 is __dense__; one with $\rho$ close to 0 is __sparse__. When $n<2$, no loop-free edges are possible, so we use the convention $\rho(\mathcal{G})=0$.

The graphs in the figure each have six vertices and seven edges. Their densities differ: the undirected graph has density $7/15$, while the directed graph has density $7/30$, because twice as many directed edges are possible.

Degree and density do not show how edges are arranged. For $n\geq4$, a path graph and a star graph can both have $n$ vertices and $n-1$ edges. The path has maximum degree 2 and diameter $n-1$; the star has maximum degree $n-1$ and diameter 2. For an undirected graph, the next four measures answer questions about distance and which vertices may be grouped together:

* __Diameter__ $\text{diam}(\mathcal{G})$: for a connected graph, the largest shortest-path distance between two vertices, measured in edges.
* __Clique number__ $\omega(\mathcal{G})$: the size of the largest set of vertices in which every pair is joined, such as the largest group of people who all know each other.
* __Chromatic number__ $\chi(\mathcal{G})$: the fewest colors needed so that no edge joins two vertices of the same color, such as the fewest time slots that schedule a set of classes with no student in two classes at once.
* __Independence number__ $\alpha(\mathcal{G})$: the size of the largest set of vertices with no edge among them, such as the most activities that can run at the same time with no conflict.

Vertex and edge counts alone do not determine these four measures. Computing them requires edge lookups and neighbor lists.

___

## How are Graphs Stored?

Let's compare three ways to store the graph: an edge list, an adjacency matrix, and an adjacency list.

An __edge list__ stores one record per edge, containing its endpoints and, when present, its weight. Text files often use this format, including the worked example in this section. Without an index, finding one edge or all neighbors of a vertex may require scanning all the records in the list. The edge list is a simple way to store a graph, but it is not efficient for algorithms that need to look up edges or neighbors.

An __adjacency matrix__ $\mathbf{A}$ for a graph with $|\mathcal{V}|$ vertices is a $|\mathcal{V}|\times|\mathcal{V}|$ matrix. The entry $a_{ij}$ in row $i$ and column $j$ describes the edge from $v_{i}$ to $v_{j}$:

* __Unweighted__: $a_{ij}=1$ if the edge exists and $a_{ij}=0$ if it does not.
* __Weighted__: $a_{ij}=w_{ij}$, the weight of the edge, if the edge exists, and $a_{ij}=0$ if it does not.

For an undirected graph the matrix is symmetric, $a_{ij} = a_{ji}$; for a directed graph it need not be. In the weighted form, a stored zero can mean either a missing edge or an edge of weight zero, so zero-weight edges require a separate marker for missing edges.

An __adjacency list__ stores one neighbor collection for each vertex. For a directed graph, each collection contains the outgoing edges from its source vertex. An unweighted list may store only the target identifiers, as in `Dict{Int64,Vector{Int64}}`. A weighted list stores the edge weight beside each target, as in `Dict{Int64,Vector{Tuple{Int64,Float64}}}`. Both forms are adjacency lists because each dictionary key is a source vertex and its value holds only that vertex's outgoing neighbors.

A flat dictionary such as `Dict{Tuple{Int64,Int64},Float64}` instead maps a known `(source, target)` pair directly to its weight. This edge-weight map gives expected constant-time lookup when both endpoints are known, but finding every neighbor of one source would require scanning all $m$ edges. A graph implementation can pair this map with an unweighted adjacency list, gaining both fast neighbor iteration and direct weight lookup.

> __What does it cost to store and use a graph?__
>
> Let $n=|\mathcal{V}|$ be the number of vertices and $m=|\mathcal{E}|$ be the number of edges. All three forms store the same graph, but the number and layout of memory entries differ. As in [the L3b lecture](../../week-03/L3b/CHEME-5800-L3b-Lecture-StacksAndQueues-Fall-2026.ipynb), $\Theta(\cdot)$ denotes a tight asymptotic growth rate, while $O(\cdot)$ states only an upper bound.
>
> * An __edge list__ stores one `(source, target, weight)` record for each edge, so its storage is $\Theta(m)$. Finding one specified edge or collecting every edge leaving $v_i$ takes $\Theta(m)$ time in the worst case because the search may inspect all $m$ records.
> * An __adjacency matrix__ stores one entry $a_{ij}$ for every ordered pair of vertices, so its storage is $\Theta(n^2)$ whether or not most pairs are connected. The entry for $(v_i,v_j)$ can be read in $\Theta(1)$ time, while finding every neighbor of $v_i$ requires scanning the full row and takes $\Theta(n)$ time.
> * An __adjacency list__ stores one neighbor collection for each vertex and one neighbor record for each directed edge, so its storage is $\Theta(n+m)$. The record may contain only a target or a `(target, weight)` pair. Accessing a vertex's vector in [a `Dict`](https://docs.julialang.org/en/v1/base/collections/#Dictionaries) takes expected $\Theta(1)$ time. Iterating over every out-neighbor of $v_i$ takes $\Theta(\deg^{\mathrm{out}}(v_i))$ time; testing for one target by a linear scan takes the same time in the worst case.
>
> Ignoring container overhead and the fixed number of fields in an edge record, storage grows as follows:
> $$
> S_{\mathrm{edge\ list}}=\Theta(m),\qquad S_{\mathrm{matrix}}=\Theta(n^2),\qquad S_{\mathrm{adjacency\ list}}=\Theta(n+m).
> $$
> For an undirected adjacency list, each edge appears in both endpoint lists, giving $n+2m$ stored items. Storage still grows as $\Theta(n+m)$.

### Storage and Access Costs

The storage bounds and access costs can be compared directly. The table uses the exact operations described above: finding one specified directed edge $(u,v)$ and listing every out-neighbor of a known source vertex $u$.

| Representation | Storage | Find $(u,v)$ | List the out-neighbors of $u$ |
|:--|:--:|:--:|:--:|
| Edge list | $\Theta(m)$ | $\Theta(m)$ | $\Theta(m)$ |
| Adjacency matrix | $\Theta(n^2)$ | $\Theta(1)$ | $\Theta(n)$ |
| Adjacency list | $\Theta(n+m)$ | $\Theta(\deg^{\mathrm{out}}(u))$ | $\Theta(\deg^{\mathrm{out}}(u))$ |

The adjacency-list edge test assumes a linear scan of the source vertex's neighbor vector. Pairing the adjacency list with the flat edge-weight map described above changes a known-pair weight lookup to expected $\Theta(1)$ time while preserving $\Theta(\deg^{\mathrm{out}}(u))$ neighbor iteration.

We choose a representation based on what task we need to do. An adjacency matrix stores all $n^2$ vertex pairs and answers one edge lookup in $\Theta(1)$ time. An adjacency list stores no entries for missing edges and scans only the out-neighbors of a vertex. Use a matrix for a dense graph with repeated edge lookups. Use a list to traverse a sparse graph. 

[The L4b traversal lab](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb) uses an adjacency list because breadth-first and depth-first search read the out-neighbors of every visited vertex.

### Worked Example: One Graph in Three Forms

The seven records in [`data/SimpleGraph.txt`](data/SimpleGraph.txt) store a directed graph with six vertices and one weight per edge. [The L4b traversal lab](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb) uses a copy of the same data. [The `read_weighted_edges(...)` function](../../../code/src/GraphRepresentation.jl) loads the edge records, [the `adjacency_list(...)` function](../../../code/src/GraphRepresentation.jl) builds the outgoing-neighbor lists, and [the `adjacency_matrix(...)` function](../../../code/src/GraphRepresentation.jl) builds the weighted matrix.

The [`adjacency_list(...)` helper](../../../code/src/GraphRepresentation.jl) deliberately builds an unweighted list for the traversal work in L4b, so this particular list keeps the out-neighbors but not their weights. That choice is not a limitation of adjacency lists: a weighted variant could store `(target, weight)` pairs. The original edge records and the matrix both keep the weights. We store the edge records in `edge_records::Vector{<:NamedTuple}`, the unweighted adjacency list in `adjacency::Dict{Int64, Vector{Int64}}`, and the matrix with its row and column order in `matrix_representation::NamedTuple`.

In [ ]:
# Build three representations of the same directed, weighted graph -
# The let block keeps temporary names inside it and returns the three notebook values.
edge_records, adjacency, matrix_representation = let
    
    # Locate and read the edge list -
    edge_path = joinpath(CHEME5800_L4A_DATA, "SimpleGraph.txt") # start from the L4a data folder, not pwd()
    records = read_weighted_edges(edge_path)                    # Vector of (source, target, weight) NamedTuples

    # Convert the edge records into two graph data structures -
    list = adjacency_list(records)                               # Dict: vertex id => sorted outgoing-neighbor ids
    matrix = adjacency_matrix(records)                           # NamedTuple: weighted matrix plus row/column vertex ids

    # Return values from the local scope -
    records, list, matrix                                       # return the three representations from let
end; # hide the full output; display the stored representations separately

The two outputs store the same directed edges in different forms. The value `vertex_order[i]` gives the vertex for row and column `i`, so the vertex IDs do not need to equal `1:n`.

In [ ]:
# Display the out-neighbor list beside the equivalent weighted matrix -
(
    adjacency = adjacency,                            # each key is a vertex; each value is its sorted out-neighbor vector
    vertex_order = matrix_representation.vertex_ids, # position i in this vector identifies matrix row and column i
    matrix = matrix_representation.matrix,            # entry (i,j) is the weight from vertex_order[i] to vertex_order[j]
)

Vertex 1 points to vertices 2 and 3, so `adjacency[1]` contains `[2, 3]` and row 1 of the matrix contains weights 10 and 100 in columns 2 and 3. Vertex 6 has an empty neighbor vector because no edge leaves it.

[The `representation_report(...)` function](../../../code/src/GraphRepresentation.jl) returns the vertex and edge counts, the directed density $|\mathcal{E}|/(|\mathcal{V}|(|\mathcal{V}|-1))$, the $|\mathcal{V}|^{2}$ matrix entries, and the $|\mathcal{V}|+|\mathcal{E}|$ adjacency-list items. We also compute storage for 100,000 vertices with ten outgoing edges per vertex, using 8 bytes per matrix entry or list item and excluding container overhead. The six-vertex report is stored in `representation::NamedTuple`.

In [ ]:
# Compare storage for the six-vertex graph and a larger sparse graph -
# The let block returns only the six-vertex report; the large-graph values stay inside the block.
representation = let
    # Measure the graph read from SimpleGraph.txt -
    report = representation_report(edge_records)                    # graph measures and storage-entry counts

    # Set the size of a larger sparse graph for the storage comparison -
    n, k = 100_000, 10                                              # n vertices, k outgoing edges each, and m = n⋅k
    bytes_per_entry = sizeof(Float64)                               # Float64 weights and Int64 ids each use 8 bytes

    # Estimate storage without counting container overhead -
    matrix_gigabytes = n^2 * bytes_per_entry / 1e9                  # n² weighted entries, converted to decimal gigabytes
    adjacency_list_megabytes = (n + n * k) * bytes_per_entry / 1e6  # n keys plus n⋅k targets, in decimal MB

    # Display the large-graph comparison and return the six-vertex report -
    println("Large sparse graph: matrix ≈ $(matrix_gigabytes) GB, adjacency list ≈ $(adjacency_list_megabytes) MB")
    report                                                         # return the six-vertex NamedTuple from let
end

For the six-vertex graph, the matrix reserves 36 entries. The directed adjacency list contains 6 vertex keys and 7 neighbor slots. For 100,000 vertices with mean out-degree 10, the matrix contains $10^{10}$ entries and occupies 80 decimal gigabytes at 8 bytes per entry. The list contains $1.1\times10^6$ items and occupies 8.8 decimal megabytes before container overhead. Matrix storage grows as $n^2$; list storage grows as $n+m$. The following tests check the six-vertex counts, density, adjacency list, and one matrix weight against the input file.

In [ ]:
# Check that every representation stores the expected graph -
@testset "graph representations" begin
    # Check the structure encoded by the input and both representations -
    @test representation.vertices == 6                       # the edge endpoints use the six vertex ids 1,...,6
    @test representation.edges == 7                          # the file contains seven directed source-to-target records
    @test representation.density ≈ 7 / 30                    # 7 observed edges divided by 6(6-1) possible loop-free edges
    @test adjacency[1] == [2, 3]                             # vertex 1 points outward to vertices 2 and 3, in sorted order
    @test matrix_representation.matrix[1, 2] == 10.0         # entry (1,2) stores the weight of edge 1 → 2

    # Check the exact storage counts used in the lecture comparison -
    @test representation.matrix_entries == 36                # a 6×6 matrix reserves one entry for every ordered pair
    @test representation.adjacency_list_entries == 13        # six dictionary keys plus one target entry per edge
end

___

## Graph Families

Let's talk about three families of graphs that are important in applications and in theory: complete graphs, bipartite graphs, and trees. Each family has a simple definition, but the members can have very different edge arrangements.

### Complete Graphs

A complete graph $K_{n}$ is a simple undirected graph on $n$ vertices with an edge between every pair of distinct vertices. Once $n$ is known, the graph is fixed except for the vertex names.

<div>
    <center>
        <img src="figs/Fig-Complete-Graph-Schematic.svg" width="900" alt="The complete graph K5 with five vertices and all ten possible edges. The four edges incident to vertex 1 are highlighted, showing that every vertex has degree 4."/>
    </center>
</div>

The figure shows $K_{5}$. The highlighted edges are the four edges incident to vertex 1. Every other vertex also has degree 4, and the graph contains all $\binom{5}{2}=10$ possible edges.

> __Counts and measures for $K_n$:__
>
> Every vertex touches the other $n-1$ vertices, so the graph $K_n$ is a regular graph of degree $n-1$. Every possible edge is present, and every pair of vertices is one edge apart. For $n\geq 2$, the edge count, density, and diameter are given by:
> $$
> \begin{align*}
> |\mathcal{E}| &= \binom{n}{2}=\frac{n(n-1)}{2}, & \deg(v_i) &= n-1,\\
> \rho(K_n) &= 1, & \operatorname{diam}(K_n) &= 1.
> \end{align*}
> $$
> The full vertex set is a clique, adjacent vertices require different colors, and an independent set can contain only one vertex. These facts give:
> $$
> \omega(K_n)=\chi(K_n)=n,\qquad \alpha(K_n)=1.
> $$
> For an algorithm whose work grows with the number of edges $|\mathcal{E}|$, the complete graph $K_n$ has the largest edge count possible for an undirected simple graph on $n$ vertices.

__What are some examples of complete graphs?__ A round-robin tournament has the edge pattern of $K_n$: every team plays every other team.  The traveling-salesman problem uses a weighted complete graph when every city pair has a travel cost and the goal is to find the tour of all cities with the least cost.

### Bipartite Graphs

A graph $\mathcal{G}=(\mathcal{V},\mathcal{E})$ is bipartite if its vertices can be split into two groups, $\mathcal{V}_{1}$ and $\mathcal{V}_{2}$, with no vertex in both groups. Every edge has one endpoint in each group, so no edge joins two vertices in the same group.

<div>
    <center>
        <img src="figs/Fig-Bipartite-Graph-Schematic.svg" width="900" alt="The complete bipartite graph K3,4 with three blue vertices in the left group and four gold vertices in the right group. All twelve edges cross between the groups, and three highlighted edges form a matching that covers the left group."/>
    </center>
</div>

The figure shows $K_{3,4}$. It has three vertices in $\mathcal{V}_{1}$, four vertices in $\mathcal{V}_{2}$, and all $3\times4=12$ edges between the two groups. The three highlighted edges do not share an endpoint, so they form a matching that covers $\mathcal{V}_{1}$. In general, let $m=|\mathcal{V}_{1}|\geq1$ and $n=|\mathcal{V}_{2}|\geq1$. The __complete bipartite graph__ $K_{m,n}$ has all $mn$ edges between the groups.

A __matching__ is a set of edges that do not share vertices. In an assignment graph, a matching that covers $\mathcal{V}_{1}$ gives every worker or student on the left a different option on the right.

> __Three tests for a bipartite graph:__
>
> For an undirected graph $\mathcal{G}$, the following are equivalent:
>
> 1. $\mathcal{G}$ is bipartite.
> 2. $\mathcal{G}$ can be colored with at most two colors, so $\chi(\mathcal{G}) \leq 2$.
> 3. $\mathcal{G}$ has no cycle of odd length.
>
> Coloring the two groups gives $1\Rightarrow2$. A two-coloring must alternate colors around a cycle, so it cannot close an odd cycle; this gives $2\Rightarrow3$. For $3\Rightarrow1$, start a breadth-first search in each connected component and color vertices by even or odd distance from the start. An edge between two vertices with the same distance parity would create an odd cycle.
>
> In $K_{m,n}$, each vertex of $\mathcal{V}_{1}$ has degree $n$, and each vertex of $\mathcal{V}_{2}$ has degree $m$. The edge count and degrees are:
> $$
> |\mathcal{E}(K_{m,n})|=mn,\qquad \deg(v)=\begin{cases}n & v\in\mathcal{V}_{1},\\ m & v\in\mathcal{V}_{2}\end{cases}.
> $$
> For $m,n\geq1$, the graph $K_{m,n}$ is regular exactly when $m=n$.
>
> __Hall's theorem__ tests whether a matching covers every vertex of $\mathcal{V}_{1}$. For a set $S\subseteq\mathcal{V}_{1}$, let $N(S)\subseteq\mathcal{V}_{2}$ be the set of all neighbors of vertices in $S$. A matching that covers $\mathcal{V}_{1}$ exists if and only if:
> $$
> |N(S)|\geq |S|\qquad\text{for every }S\subseteq\mathcal{V}_{1}.
> $$
> In words, every set of vertices on the left must have at least as many neighbors on the right. A matching is __perfect__ when it covers both groups; this requires $|\mathcal{V}_{1}|=|\mathcal{V}_{2}|$.
>
> List the vertices of $\mathcal{V}_{1}$ before the vertices of $\mathcal{V}_{2}$. The adjacency matrix then has the block form:
> $$
> \mathbf{A}=\begin{bmatrix}\mathbf{0} & \mathbf{B}\\ \mathbf{B}^{\top} & \mathbf{0}\end{bmatrix},
> $$
> where the $m\times n$ __biadjacency matrix__ $\mathbf{B}$ stores the edges between the two groups. The two diagonal blocks are zero because no edge stays within a group.

The coloring test can be carried out with a graph traversal:

1. Mark every vertex uncolored.
2. Choose an uncolored vertex, give it color 1, and run breadth-first or depth-first search through its connected component.
3. Give each uncolored neighbor the opposite color of the current vertex. If an edge joins two vertices with the same color, stop: the graph is not bipartite.
4. Repeat from another uncolored vertex until every connected component has been checked. If no conflict occurs, the two color classes are $\mathcal{V}_{1}$ and $\mathcal{V}_{2}$.

With an undirected adjacency list, this test visits each vertex once and reads each edge from both endpoint lists. Its running time is $\Theta(|\mathcal{V}|+|\mathcal{E}|)$. [The L4b lab](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb) develops the breadth-first and depth-first searches used by this test.

Bipartite graphs model links between two types of objects: workers and jobs, students and courses, users and items, regulatory proteins and the genes they control, or species and habitats. An adjacency list stores these graphs in the same form as other graphs. In a matrix, storing $\mathbf{B}$ is enough because it determines $\mathbf{B}^{\top}$ and the two zero blocks.

### Trees

A tree $\mathcal{T}=(\mathcal{V},\mathcal{E})$ is a connected undirected graph with no cycles. Connectivity gives at least one path between each pair of vertices. If two such paths existed, their union would contain a cycle, so the path is unique. Removing any edge disconnects the tree. Adding an edge between two vertices that are not already adjacent creates one cycle.

> __Six tests for a tree:__
>
> For an undirected graph $\mathcal{G}$ with $n\geq1$ vertices, the following statements are equivalent:
>
> 1. $\mathcal{G}$ is a tree.
> 2. $\mathcal{G}$ is connected and has exactly $n-1$ edges.
> 3. $\mathcal{G}$ has no cycle and has exactly $n-1$ edges.
> 4. $\mathcal{G}$ is connected, and removing any edge disconnects it.
> 5. $\mathcal{G}$ has no cycle, and adding any missing edge creates exactly one cycle.
> 6. Any two vertices of $\mathcal{G}$ are joined by exactly one path.
>
> Any one of these tests also shows that a tree has $n-1$ edges. For $n\geq2$, its edge count and density are:
> $$
> |\mathcal{E}|=n-1,\qquad \rho(\mathcal{G})=\frac{n-1}{n(n-1)/2}=\frac{2}{n}.
> $$
> The density is $2/n$, so it tends to zero as $n$ grows even though the tree remains connected.

Choose one vertex as the __root__. Every other vertex has one __parent__: the next vertex on its path to the root. A vertex may have zero or more __children__. A vertex with no children is a __leaf__. The __depth__ of a vertex is the number of edges from the root to that vertex. The __height__ of the rooted tree is its largest vertex depth.

<div>
    <center>
        <img src="figs/Fig-General-Tree-Schematic.svg" width="880" alt="A rooted tree with root r at depth 0, internal vertices at depths 1 and 2, leaves at depths 2 and 3, and height 3."/>
    </center>
</div>

The figure shows a rooted tree of height 3. Rooted trees model file systems, organization charts, and function calls because each non-root vertex has one parent.

Every connected undirected graph $\mathcal{G}$ contains a __spanning tree__: a subgraph $\mathcal{T}$ with every vertex of $\mathcal{G}$ and $n-1$ edges that keep the vertices connected. If a graph has no cycles but is not connected, each connected component is a tree. Their union is a __forest__.

A standard tree dynamic program visits each vertex once and finds a largest independent set in $\Theta(n)$ time, while the same problem is NP-hard on a general graph.

___

## Looking Ahead: Traversal and Shortest Paths

A graph traversal starts at one vertex, reads its neighbor list, and repeats this step for each vertex it reaches. It stops when no unvisited reachable vertex remains. We use an adjacency list so the traversal reads only existing neighbors.

In [the L4b lab](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb), we use the same six-vertex directed graph. Breadth-first search (BFS) uses a queue and visits vertices in layers by their distance from the start. Depth-first search (DFS) follows one branch before it backtracks. The edge weights do not affect either visit order, so both algorithms use the unweighted `Dict{Int64,Vector{Int64}}` form. Each algorithm runs in $\Theta(|\mathcal{V}_r|+|\mathcal{E}_r|)$ time, where $\mathcal{V}_r$ and $\mathcal{E}_r$ denote the vertices and outgoing edges reached from the chosen start.

[The L4c shortest-path lecture](../L4c/CHEME-5800-L4c-Lecture-ShortestPathAlgorithms-Fall-2026.ipynb) needs the weights. [The Dijkstra implementation](../../../code/src/ShortestPathAlgorithms.jl) groups [`WeightedEdge` records](../../../code/src/ShortestPathAlgorithms.jl) by source in a `Dict{Int64,Vector{WeightedEdge}}`. Iterating over one source's vector then provides both `edge.target` and `edge.weight` for each outgoing edge. Bellman–Ford instead uses the edge-list form because each relaxation pass scans every edge.

___

## Summary

A graph states which pairs of vertices are joined. Its storage form sets the cost of looking up one edge and listing the neighbors of one vertex.

> __Key Takeaways:__
>
> * __Count vertices and edges:__ Vertex degree counts the edges that touch a vertex, and graph density is the fraction of possible edges that are present. In an undirected graph, summing the vertex degrees counts each edge twice.
> * __Recognize three graph families:__ A complete graph joins every pair of distinct vertices, while a bipartite graph separates its vertices into two groups and places every edge between the groups. A tree is connected, contains no cycle, and has one path between each pair of vertices.
> * __Match storage to the operation:__ An adjacency matrix reserves an entry for every ordered pair of vertices and supports constant-time edge lookup. A directed adjacency list stores only existing outgoing edges; each neighbor record may contain only a target identifier or may also carry the edge weight.

[The L4b lab](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb) uses an unweighted adjacency list for breadth-first and depth-first traversal. [The L4c lecture](../L4c/CHEME-5800-L4c-Lecture-ShortestPathAlgorithms-Fall-2026.ipynb) uses a weighted adjacency list in Dijkstra's algorithm so each relaxation step can read the target and weight of an outgoing edge.

___